In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb


In [2]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df = pd.concat([df_2024, df_2025], ignore_index=True)

In [3]:
df.head()
df= df.drop(columns="Unnamed: 0")

In [4]:


id_cols = ['player_id', 'player', 'position', 'team', 'season', 'week',
           'date', 'opponent', 'atVs']

context_cols = ['opp_def_rank', 'team_pass_rank', 'team_rush_rank',
                'stadium', 'roof', 'surface', 'temp', 'wind', 'location',
                'gametime']

target_cols = ['fantasy_points', 'projected_points']



rushing = ['rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
           'rushingTouchdowns', 'longRushing']
receiving = ['receptions', 'receivingTargets', 'receivingYards',
             'yardsPerReception', 'receivingTouchdowns', 'longReception']
ball_security = ['fumbles', 'fumblesLost']
passing = ['completions', 'passingAttempts', 'passingYards', 'completionPct',
           'yardsPerPassAttempt', 'passingTouchdowns', 'interceptions',
           'longPassing', 'sacks', 'QBRating', 'adjQBR']
kicking = ['fieldGoalsMade', 'fieldGoalAttempts', 'fieldGoalPct',
           'longFieldGoalMade', 'fieldGoalsMadeYardsAverage',
           'fieldGoalsMade1_19', 'fieldGoalAttempts1_19',
           'fieldGoalsMade20_29', 'fieldGoalAttempts20_29',
           'fieldGoalsMade30_39', 'fieldGoalAttempts30_39',
           'fieldGoalsMade40_49', 'fieldGoalAttempts40_49',
           'fieldGoalsMade50', 'fieldGoalAttempts50',
           'extraPointsMade', 'extraPointAttempts', 'totalKickingPoints']

base = id_cols + context_cols + target_cols

position_cols = {
    'QB': base + passing + rushing + ball_security,
    'WR': base + receiving + rushing + ball_security,
    'TE': base + receiving + rushing + ball_security,
    'K':  base + kicking,
    "RB": base + rushing + receiving + ball_security
}

pos_dfs = {pos: df.loc[df['position'] == pos, cols].copy()
           for pos, cols in position_cols.items()}

qb_df, wr_df, te_df, k_df, rb_df = (pos_dfs[p] for p in ['QB', 'WR', 'TE', 'K', 'RB'])

In [5]:
## combined data frames based on cluster
qb_te = pd.concat([qb_df, te_df])
#qb_te.head()


In [6]:
wr_k = pd.concat([wr_df, k_df])

In [7]:
# now we need to sort each data frame in order
rb_df = rb_df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
qb_te = qb_te.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
wr_k = wr_k.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)


In [8]:
## need to shift by -1 to create target of next weeks fantasy score
g_1 = rb_df.groupby(['player_id', "season"])
rb_df["target_fantasy_points"] = g_1["fantasy_points"].shift(-1)
rb_df['next_week_opponent'] = g_1['opponent'].shift(-1)
rb_df['next_week_atVs'] = g_1['atVs'].shift(-1)
rb_df['next_week_projected'] = g_1['projected_points'].shift(-1) 

g_2 = qb_te.groupby(['player_id', "season"])
qb_te["target_fantasy_points"] = g_2["fantasy_points"].shift(-1)
qb_te['next_week_opponent'] = g_2['opponent'].shift(-1)
qb_te['next_week_atVs'] = g_2['atVs'].shift(-1)
qb_te['next_week_projected'] = g_2['projected_points'].shift(-1) 

g_3 = wr_k.groupby(['player_id', "season"])
wr_k["target_fantasy_points"] = g_3["fantasy_points"].shift(-1)
wr_k["next_week_opponent"] = g_3["opponent"].shift(-1)
wr_k["next_week_atVs"] = g_3["atVs"].shift(-1)
wr_k["next_week_projected"] = g_3["projected_points"].shift(-1)
for col in context_cols:
    rb_df[f"next_week_{col}"] = g_1[col].shift(-1)
    qb_te[f"next_week_{col}"] = g_2[col].shift(-1)
    wr_k[f"next_week_{col}"] = g_3[col].shift(-1)


In [11]:
## encode categorical columns
shared_num = ['opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'temp', 'wind', 'week']
shared_cat = ['position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
                    'next_week_roof', 'next_week_location']
target = "target_fantasy_points"
past_cols = ["fantasy_points", "opponent", "atVs", "projected_points"]

# rb_df = rb_df.drop(columns = past_cols)
# qb_te = qb_te.drop(columns = past_cols)
# wr_k = wr_k.drop(columns = past_cols)





In [ ]:
#rb_df.columns
rb_df[shared_cat] = rb_df[shared_cat].astype("category")
qb_te[shared_cat] = qb_te[shared_cat].astype("category")
wr_k[shared_cat] = wr_k[shared_cat].astype("category")




TypeError: 'CategoricalDtype' object is not callable

In [20]:
train_data_1 = rb_df[(rb_df["season"] == 2024) | ((rb_df["season"] == 2025) & (rb_df["week"] < 7))]
train_data_2 = qb_te[(qb_te["season"] == 2024) | ((qb_te["season"] == 2025) & (qb_te["week"] < 7))]
train_data_3 = wr_k[(wr_k["season"] == 2024) | ((wr_k["season"] == 2025) & (wr_k["week"] < 7))]

val_1 = rb_df[(rb_df["season"] == 2025) & (rb_df["week"].between(7,9))]
val_2 = qb_te[(qb_te["season"] == 2025) & (qb_te["week"].between(7,9))]
val_3 = wr_k[(wr_k["season"] == 2025) & (wr_k["week"].between(7,9))]

test_1 =  rb_df[(rb_df["season"] == 2025) & (rb_df["week"] >= 10)]
test_2 =  qb_te[(qb_te["season"] == 2025) & (qb_te["week"] >= 10)]
test_3 = wr_k[(wr_k["season"] ==2025) & (wr_k["week"] >= 10)]


def clean(X, y):
    m = y.notna() & np.isfinite(y)
    return X[m], y[m]



In [24]:
common = list(set(rb_df.columns) & set(qb_te.columns) & set(wr_k.columns))
common

['next_week_projected',
 'next_week_team_pass_rank',
 'next_week_opponent',
 'next_week_opp_def_rank',
 'next_week_stadium',
 'opp_def_rank',
 'surface',
 'stadium',
 'longRushing',
 'team_pass_rank',
 'rushingTouchdowns',
 'wind',
 'gametime',
 'player',
 'receivingYards',
 'target_fantasy_points',
 'next_week_location',
 'rushingYards',
 'team',
 'roof',
 'receivingTargets',
 'rushingAttempts',
 'receivingTouchdowns',
 'next_week_temp',
 'next_week_surface',
 'date',
 'fumbles',
 'player_id',
 'yardsPerRushAttempt',
 'next_week_atVs',
 'next_week_gametime',
 'fumblesLost',
 'position',
 'next_week_wind',
 'receptions',
 'yardsPerReception',
 'next_week_roof',
 'temp',
 'location',
 'longReception',
 'next_week_team_rush_rank',
 'season',
 'week',
 'team_rush_rank']

In [18]:
irrelevant_columns = [
    'player_id', 'player', 'opponent',
    'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
    'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
    '_set', 'season',
    'med_proj_pts',
    'projected_points',
    'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
    'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected' 
]
irrelevant = list(set(common) & set(irrelevant_columns))

In [21]:

def get_data(x):
    x_train = x.loc[:, ~x.columns.isin(irrelevant)]
    y_train = x[target]

    x_train , y_train = clean(x_train, y_train)

    return x_train, y_train


X1_train, y1_train = get_data(train_data_1)
X2_train, y2_train = get_data(train_data_2)
X3_train, y3_train = get_data(train_data_3)

X1_val, y1_val = get_data(val_1)
X2_val, y2_val = get_data(val_2)
X3_val, y3_val = get_data(val_3)

X1_test, y1_test = get_data(test_1)
X2_test, y2_test = get_data(test_2)
X3_test, y3_test = get_data(test_3)

In [22]:
model1 = xgb.XGBRegressor(
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)



In [23]:
model1.fit(
    X1_train, y1_train,
    eval_set=[(X1_val, y1_val)],
    verbose=50 #how often to print validation score 
)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:position: category, team: str, date: str, next_week_opponent: category, next_week_atVs: category, next_week_stadium: str, next_week_roof: category, next_week_surface: category, next_week_location: category, next_week_gametime: str